<a href="https://colab.research.google.com/github/awildt01/Credit-Scoring/blob/main/3__select_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zumanenfassung

## **Verwendete Methode zur Feature-Auswahl**

In diesem Schritt wird eine Kombination aus manueller Feature-Selektion und der Vermeidung der Dummy-Variablen-Falle angewendet.

- **Manuelle Feature-Selektion**
Die Auswahl der über 100 genutzten Variablen basiert nicht auf einem automatisierten Algorithmus, sondern auf einer vorangegangenen Analyse (z. B. explorative Datenanalyse, Berechnung von Information Value oder Weight of Evidence). Ein Analyst hat entschieden, welche Merkmale für das Modell am relevantesten sind.

- **Vermeidung der Dummy-Variablen-Falle**
Bei kategorialen Variablen mit k Kategorien entstehen nach dem One-Hot-Encoding k Dummy-Variablen. Werden alle verwendet, führt dies zu perfekter **Multikollinearität**, was insbesondere bei linearer und logistischer Regression problematisch ist.

**Lösung:** Aus jeder Gruppe von Dummy-Variablen wird eine Kategorie entfernt. Diese Referenzkategorie dient als Vergleichsbasis.
Beispiel: Im Code wird term:60 (Laufzeit: 60 Monate) entfernt. Der Koeffizient von term:36 misst dann den Unterschied im Kreditrisiko im Vergleich zur Referenz term:60.

Fazit:
Die eigentliche Feature-Auswahl erfolgt bereits vor diesem Schritt. Der hier gezeigte Code bereitet die ausgewählten Variablen für die Modellierung vor, indem er Multikollinearität durch die Wahl einer Referenzkategorie verhindert.

## **Vorbereitung der Eingabedaten: Feature-Auswahl und Referenzkategorien für die logistische Regression**

**1- Auswahl der Features (Variablenauswahl):** Zuerst wird ein neuer DataFrame **inputs_train_with_ref_cat** erstellt. Dieser enthält nur eine spezifische Auswahl von Spalten (Features) aus dem ursprünglichen Trainingsdatensatz **loan_data_inputs_train**. Die ausgewählten Spalten sind bereits in "Dummy-Variablen" umgewandelt worden. Das bedeutet, dass kategoriale Variablen wie grade (Kreditwürdigkeitsstufe) oder home_ownership (Wohneigentum-Status) in mehrere binäre (0/1) Spalten aufgeteilt wurden.

**2- Definition der Referenzkategorien:** Eine Liste namens **ref_categories** wird erstellt. Diese Liste enthält die Namen von Dummy-Variablen, die als "Referenzkategorien" dienen sollen. Für jede ursprüngliche kategoriale Variable (z.B. grade) wird eine ihrer Dummy-Variablen (hier grade:G) in diese Liste aufgenommen.

**3- Entfernen der Referenzkategorien:** Die in der **ref_categories-Liste** definierten Spalten werden aus dem DataFrame inputs_train_with_ref_cat entfernt. Das Ergebnis ist ein neuer, finaler DataFrame inputs_train, der für das Modelltraining verwendet wird.
Zusammenfassend lässt sich sagen: Der Code selektiert eine Reihe von vorverarbeiteten Features und entfernt dann aus jeder kategorialen Gruppe eine Referenzkategorie, um Multikollinearität zu vermeiden.

Zusammenfassend ist die Definition einer Referenzkategorie eine technische Notwendigkeit, um **Multikollinearität zu vermeiden**, und gleichzeitig ein mächtiges Werkzeug für die Interpretation, da sie den Bezugsrahmen für die Bewertung aller anderen Kategorien festlegt.

## **Was ist eine Referenzkategorie?**

Eine Referenzkategorie ist die Basiskategorie, mit der alle anderen Kategorien einer kategorialen Variable verglichen werden, wenn sie in einem statistischen Modell (wie der logistischen oder linearen Regression) verwendet wird.
Stellen Sie sich vor, Sie möchten den Einfluss der home_ownership (Wohneigentum-Status) auf das Kreditrisiko untersuchen. Diese Variable hat drei mögliche Ausprägungen (Kategorien): MORTGAGE (Hypothek), OWN (Eigentum) und RENT (Miete).
Ein Computer kann mit den Textwerten "MORTGAGE" oder "RENT" nicht direkt rechnen. Deshalb wandeln wir sie in Dummy-Variablen um. Das sieht so aus:

| home_ownership | home_ownership:MORTGAGE | home_ownership:OWN | home_ownership:RENT |
|----------------|-------------------------|--------------------|---------------------|
| MORTGAGE       | 1                       | 0                  | 0                   |
| OWN            | 0                       | 1                  | 0                   |
| RENT           | 0                       | 0                  | 1                   |



**Das Problem: Die "Dummy-Variablen-Falle" (Multikollinearität)**

Wenn wir alle drei Dummy-Variablen in ein Regressionsmodell aufnehmen, stoßen wir auf ein Problem namens perfekte Multikollinearität. Das bedeutet, dass eine Variable perfekt aus den anderen vorhergesagt werden kann.
  - Beispiel: Wenn home_ownership:MORTGAGE = 0 und home_ownership:OWN = 0 ist, dann muss home_ownership:RENT = 1 sein. Es gibt keine andere Möglichkeit.

Diese redundante Information verwirrt das Modell. Es kann keine eindeutigen Koeffizienten für jede Kategorie berechnen, weil ihre Effekte nicht voneinander zu trennen sind. Mathematisch führt dies zu einem Fehler.



**Die Lösung:**

Eine Kategorie als Referenzpunkt festlegen
Um dieses Problem zu lösen, lassen wir einfach eine der Dummy-Variablen weg. Die weggelassene Kategorie wird zur Referenzkategorie.
In Ihrem Code wurde für home_ownership die Kategorie RENT_OTHER_NONE_ANY als Referenz gewählt. Nehmen wir an, wir vereinfachen das auf RENT. Dann sieht der Datensatz, den das Modell tatsächlich verwendet, so aus:

| home_ownership | home_ownership:MORTGAGE | home_ownership:OWN |
|----------------|-------------------------|--------------------|
| MORTGAGE       | 1                       | 0                  |
| OWN            | 0                       | 1                  |
| RENT           | 0                       | 0                  |

**Wie wird das jetzt interpretiert?**

- Ein Kreditnehmer mit MORTGAGE wird durch home_ownership:MORTGAGE = 1 identifiziert.
- Ein Kreditnehmer mit OWN wird durch home_ownership:OWN = 1 identifiziert.
- Ein Kreditnehmer mit RENT (die Referenzkategorie) wird dadurch identifiziert, dass beide anderen Spalten 0 sind. Er ist der "Standardfall".

**Die Bedeutung im Modell:** Alles ist relativ

Wenn das Modell trainiert wird, berechnet es Koeffizienten (Gewichte) für home_ownership:MORTGAGE und home_ownership:OWN. Diese Koeffizienten beschreiben den Unterschied im Effekt im Vergleich zur Referenzkategorie (RENT).
Angenommen, das Modell liefert folgende (fiktive) Koeffizienten für die Wahrscheinlichkeit eines Kreditausfalls:
- home_ownership:MORTGAGE: -0.25
- home_ownership:OWN: -0.10

**Interpretation:**

- Der negative Koeffizient von -0.25 für MORTGAGE bedeutet: Kreditnehmer mit einer Hypothek haben ein geringeres Kreditausfallrisiko als Kreditnehmer, die zur Miete wohnen.
- Der Koeffizient von -0.10 für OWN bedeutet: Kreditnehmer mit Eigenheim haben ebenfalls ein geringeres Risiko als Mieter, aber der risikomindernde Effekt ist nicht so stark wie bei Hypothekennehmern.

Die Referenzkategorie (RENT) hat keinen eigenen Koeffizienten. Ihr Effekt ist sozusagen im "Grundrauschen" des Modells (dem Intercept oder Achsenabschnitt) enthalten. Sie ist der Nullpunkt, von dem aus die anderen Effekte gemessen werden.

## **Wie wählt man die Referenzkategorie aus?**

Die Wahl der Referenzkategorie ist eine strategische Entscheidung. Übliche Vorgehensweisen sind:
1. **Die häufigste Kategorie:** Das macht die Interpretation der anderen Koeffizienten oft intuitiv, da sie den Unterschied zur "normalsten" Gruppe zeigen.
2. **Die Kategorie mit dem geringsten Risiko (oder Erfolg):** Wenn man z.B. grade:G (die schlechteste Bonitätsnote) als Referenz wählt, werden alle anderen Koeffizienten (grade:A, grade:B etc.) positiv sein und zeigen, um wie viel besser diese Noten im Vergleich zur schlechtesten sind. Dies ist in Ihrem Code der Fall und eine sehr gängige Praxis in der Kreditrisikomodellierung.
3. **Eine logische "Basis"-Kategorie:** Zum Beispiel könnte man bei einer medizinischen Studie die "Placebo"-Gruppe als Referenz wählen.

# Import Libraries

In [1]:
import numpy as np
import pandas as pd

from google.colab import drive
import os

In [2]:
# Google Drive mounten
drive.mount('/content/drive')

# Zielordner definieren
folder_path = '/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks'
os.makedirs(folder_path, exist_ok=True)  # Ordner erstellen, falls nicht vorhanden

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
loan_data_inputs_train = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/loan_data_inputs_train.csv', index_col = 0,)

In [4]:
loan_data_targets_train = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/loan_data_targets_train.csv', index_col = 0,header=0)

In [5]:
loan_data_inputs_test = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/loan_data_inputs_test.csv',index_col = 0,)

In [6]:
loan_data_targets_test = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/loan_data_targets_test.csv',index_col = 0, header=0)

# Loading the Data and Selecting the Features

In [7]:
#loan_data_inputs_train = pd.read_csv('loan_data_inputs_train.csv', index_col = 0)
# loan_data_targets_train = pd.read_csv('loan_data_targets_train.csv', index_col = 0, header = None)
# loan_data_inputs_test = pd.read_csv('loan_data_inputs_test.csv', index_col = 0)
# loan_data_targets_test = pd.read_csv('loan_data_targets_test.csv', index_col = 0, header = None)

### Explore Data

In [8]:
pd.set_option('display.max_info_rows', 300)
loan_data_inputs_train.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
427211,292984,29074404,31607606,11250,11250,11250.0,60 months,16.99,279.54,D,...,0,0,0,1,0,0,0,0,0,0
206088,57793,9026077,10838105,8000,8000,8000.0,36 months,18.55,291.44,D,...,0,1,0,1,0,0,0,0,0,0
136020,140700,4865088,6157358,6000,6000,6000.0,36 months,10.16,194.06,B,...,0,0,0,1,0,0,0,0,0,0
412305,297571,28764454,31297687,8000,8000,8000.0,36 months,9.17,255.04,B,...,0,0,0,1,0,0,0,0,0,0
36159,5264,985083,1208540,5000,5000,5000.0,36 months,11.71,165.38,B,...,0,0,0,1,0,0,0,0,0,0


In [9]:
loan_data_targets_train.shape

(373028, 1)

In [10]:
loan_data_inputs_train.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
427211,292984,29074404,31607606,11250,11250,11250.0,60 months,16.99,279.54,D,...,0,0,0,1,0,0,0,0,0,0
206088,57793,9026077,10838105,8000,8000,8000.0,36 months,18.55,291.44,D,...,0,1,0,1,0,0,0,0,0,0
136020,140700,4865088,6157358,6000,6000,6000.0,36 months,10.16,194.06,B,...,0,0,0,1,0,0,0,0,0,0
412305,297571,28764454,31297687,8000,8000,8000.0,36 months,9.17,255.04,B,...,0,0,0,1,0,0,0,0,0,0
36159,5264,985083,1208540,5000,5000,5000.0,36 months,11.71,165.38,B,...,0,0,0,1,0,0,0,0,0,0


In [11]:
loan_data_targets_train.head()

,good_bad
427211,1
206088,1
136020,1
412305,1
36159,0


In [12]:
loan_data_inputs_test.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
362514,343487,21211587,23514367,6400,6400,6400.0,36 months,8.39,201.71,A,...,1,0,0,1,0,0,0,0,0,0
288564,415939,12656479,14658633,18700,18700,18700.0,36 months,18.92,684.72,D,...,0,0,0,1,0,0,0,0,0,0
213591,65794,8618870,10390702,16000,16000,16000.0,36 months,6.03,486.97,A,...,0,0,0,1,0,0,0,0,0,0
263083,426133,13045554,15077755,15000,15000,15000.0,60 months,17.57,377.41,D,...,1,0,0,1,0,0,0,0,0,0
165001,100410,6826607,8448689,16000,16000,16000.0,36 months,15.22,556.38,C,...,0,0,0,1,0,0,0,0,0,0


In [27]:
loan_data_targets_test.head()

,good_bad
362514,1
288564,1
213591,1
263083,1
165001,1


In [28]:
for i, col in enumerate(loan_data_inputs_train.columns, 1):
    print(f"{i}. {col}")

1. Unnamed: 0
2. id
3. member_id
4. loan_amnt
5. funded_amnt
6. funded_amnt_inv
7. term
8. int_rate
9. installment
10. grade
11. sub_grade
12. emp_title
13. emp_length
14. home_ownership
15. annual_inc
16. verification_status
17. issue_d
18. loan_status
19. pymnt_plan
20. url
21. desc
22. purpose
23. title
24. zip_code
25. addr_state
26. dti
27. delinq_2yrs
28. earliest_cr_line
29. inq_last_6mths
30. mths_since_last_delinq
31. mths_since_last_record
32. open_acc
33. pub_rec
34. revol_bal
35. revol_util
36. total_acc
37. initial_list_status
38. out_prncp
39. out_prncp_inv
40. total_pymnt
41. total_pymnt_inv
42. total_rec_prncp
43. total_rec_int
44. total_rec_late_fee
45. recoveries
46. collection_recovery_fee
47. last_pymnt_d
48. last_pymnt_amnt
49. next_pymnt_d
50. last_credit_pull_d
51. collections_12_mths_ex_med
52. mths_since_last_major_derog
53. policy_code
54. application_type
55. annual_inc_joint
56. dti_joint
57. verification_status_joint
58. acc_now_delinq
59. tot_coll_amt
60. 

### Selecting the Features

In [29]:
inputs_train_with_ref_cat = loan_data_inputs_train.loc[: , ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',

'home_ownership:RENT_OTHER_NONE_ANY',
'home_ownership:OWN',
'home_ownership:MORTGAGE',

'addr_state:ND',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'addr_state:OK_LA_NC_NM_MO_VA_NJ',
'addr_state:MD_TN_AZ_PA_MI',
'addr_state:DE_AR',
'addr_state:UT_MN_OH_IN_GA_ RI_WA',
'addr_state:OR_KY_MA',
'addr_state:MT_MS_SD',
'addr_state:WI_IL_CT_AK',
'addr_state:CO_SC',
'addr_state:KS_NH_WV_VT_ID',
'addr_state:WY_DC_ME',

'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',

'purpose:educ__sm_b__wedd__ren_en__mov__house',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:oth__med__vacation',
'purpose:major_purch__car__home_impr',

'initial_list_status:f',
'initial_list_status:w',

'term:36',
'term:60',

'emp_length:0',
'emp_length:1',
'emp_length:2-4',
'emp_length:5-6',
'emp_length:7-9',
'emp_length:10',

'mths_since_issue_d:<38',
'mths_since_issue_d:38-39',
'mths_since_issue_d:40-41',
'mths_since_issue_d:42-48',
'mths_since_issue_d:49-52',
'mths_since_issue_d:53-64',
'mths_since_issue_d:65-84',
'mths_since_issue_d:>84',

'int_rate:<9.548',
'int_rate:9.548-12.025',
'int_rate:12.025-15.74',
'int_rate:15.74-20.281',
'int_rate:>20.281',

'mths_since_earliest_cr_line:<140',
'mths_since_earliest_cr_line:141-164',
'mths_since_earliest_cr_line:165-247',
'mths_since_earliest_cr_line:248-270',
'mths_since_earliest_cr_line:271-352',
'mths_since_earliest_cr_line:>352',


'delinq_2yrs:0',
'delinq_2yrs:1-3',
'delinq_2yrs:4-16',
'delinq_2yrs:>=17',


'inq_last_6mths:0',
'inq_last_6mths:1-2',
'inq_last_6mths:3-6',
'inq_last_6mths:>6',


'open_acc:0',
'open_acc:1-3',
'open_acc:4-12',
'open_acc:13-17',
'open_acc:18-22',
'open_acc:23-25',
'open_acc:26-30',
'open_acc:>=31',

'pub_rec:0-2',
'pub_rec:3-4',
'pub_rec:>=5',

'total_acc:0-15',
'total_acc:16-70',
'total_acc:71-90',
'total_acc:>90',

'acc_now_delinq:0',
'acc_now_delinq:>=1',

'total_rev_hi_lim:<=5K',
'total_rev_hi_lim:5K-10K',
'total_rev_hi_lim:10K-20K',
'total_rev_hi_lim:20K-30K',
'total_rev_hi_lim:30K-40K',
'total_rev_hi_lim:40K-55K',
'total_rev_hi_lim:55K-95K',



'annual_inc:<20K',
'annual_inc:20K-30K',
'annual_inc:30K-40K',
'annual_inc:40K-50K',
'annual_inc:50K-60K',
'annual_inc:60K-70K',
'annual_inc:70K-80K',
'annual_inc:80K-90K',
'annual_inc:90K-100K',
'annual_inc:100K-120K',
'annual_inc:120K-140K',
'annual_inc:>140K',

'dti:<=1.4',
'dti:1.4-3.5',
'dti:3.5-7.7',
'dti:7.7-10.5',
'dti:10.5-16.1',
'dti:16.1-20.3',
'dti:20.3-21.7',
'dti:21.7-22.4',
'dti:22.4-35',
'dti:>35',


'mths_since_last_delinq:Missing',
'mths_since_last_delinq:0-3',
'mths_since_last_delinq:4-30',
'mths_since_last_delinq:31-56',
'mths_since_last_delinq:>=57',

'mths_since_last_record:0-2',
'mths_since_last_record:3-20',
'mths_since_last_record:21-31',
'mths_since_last_record:32-80',
'mths_since_last_record:81-86',
'mths_since_last_record:>86',

]]

In [30]:
# Here we store the names of the reference category dummy variables in a list.
ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
#'addr_state:ND_NE_IA_NV_FL_HI_AL',
'addr_state:ND',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'initial_list_status:f',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'delinq_2yrs:0',
'inq_last_6mths:>6',
'open_acc:0',
'pub_rec:0-2',
'total_acc:0-15',
'acc_now_delinq:0',
'total_rev_hi_lim:<=5K',
'annual_inc:<20K',
'dti:>35',
'mths_since_last_delinq:0-3',
'mths_since_last_record:0-2']

In [39]:
pd.set_option('display.max_info_rows', 100)
inputs_train = inputs_train_with_ref_cat.drop(ref_categories, axis = 1)
# From the dataframe with input variables, we drop the variables with variable names in the list with reference categories.
inputs_train.sample(100)

,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:OWN,home_ownership:MORTGAGE,addr_state:ND_NE_IA_NV_FL_HI_AL,addr_state:OK_LA_NC_NM_MO_VA_NJ,...,dti:22.4-35,mths_since_last_delinq:Missing,mths_since_last_delinq:4-30,mths_since_last_delinq:31-56,mths_since_last_delinq:>=57,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
372480,False,False,False,True,False,False,False,False,0,0,...,0,1,0,0,0,0,0,0,0,0
57781,False,True,False,False,False,False,False,False,1,0,...,0,1,0,0,0,0,0,0,0,0
29824,False,True,False,False,False,False,True,False,0,0,...,0,1,0,0,0,0,0,0,0,0
399016,False,False,True,False,False,False,False,True,0,0,...,0,0,1,0,0,0,0,0,0,0
214453,False,False,False,True,False,False,False,True,0,0,...,0,1,0,0,0,0,0,0,0,1
274390,False,False,False,True,False,False,False,True,0,0,...,0,0,1,0,0,0,0,0,0,0
322372,False,False,True,False,False,False,False,False,0,1,...,1,0,1,0,0,0,0,1,0,0
271935,False,False,True,False,False,False,False,False,0,0,...,0,0,0,0,1,0,0,0,0,0
284892,True,False,False,False,False,False,False,False,1,0,...,1,1,0,0,0,0,0,0,0,0
4842,False,False,True,False,False,False,False,True,0,1,...,0,0,0,1,0,0,0,0,0,0


In [40]:
for i, col in enumerate(inputs_train.columns, 1):
    print(f"{i}. {col}")

1. grade:A
2. grade:B
3. grade:C
4. grade:D
5. grade:E
6. grade:F
7. home_ownership:OWN
8. home_ownership:MORTGAGE
9. addr_state:ND_NE_IA_NV_FL_HI_AL
10. addr_state:OK_LA_NC_NM_MO_VA_NJ
11. addr_state:MD_TN_AZ_PA_MI
12. addr_state:DE_AR
13. addr_state:UT_MN_OH_IN_GA_ RI_WA
14. addr_state:OR_KY_MA
15. addr_state:MT_MS_SD
16. addr_state:WI_IL_CT_AK
17. addr_state:CO_SC
18. addr_state:KS_NH_WV_VT_ID
19. addr_state:WY_DC_ME
20. verification_status:Not Verified
21. verification_status:Source Verified
22. purpose:credit_card
23. purpose:debt_consolidation
24. purpose:oth__med__vacation
25. purpose:major_purch__car__home_impr
26. initial_list_status:w
27. term:36
28. emp_length:1
29. emp_length:2-4
30. emp_length:5-6
31. emp_length:7-9
32. emp_length:10
33. mths_since_issue_d:<38
34. mths_since_issue_d:38-39
35. mths_since_issue_d:40-41
36. mths_since_issue_d:42-48
37. mths_since_issue_d:49-52
38. mths_since_issue_d:53-64
39. mths_since_issue_d:65-84
40. int_rate:<9.548
41. int_rate:9.548-12.

In [41]:
import seaborn as sns
import matplotlib.pyplot as plt

matrix = inputs_train.corr()

matrix

,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,home_ownership:OWN,home_ownership:MORTGAGE,addr_state:ND_NE_IA_NV_FL_HI_AL,addr_state:OK_LA_NC_NM_MO_VA_NJ,...,dti:22.4-35,mths_since_last_delinq:Missing,mths_since_last_delinq:4-30,mths_since_last_delinq:31-56,mths_since_last_delinq:>=57,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
grade:A,1.000000,-0.281766,-0.264622,-0.193788,-0.126141,-0.074456,-0.005355,0.065822,-0.011226,-0.001681,...,-0.086285,0.128550,-0.089807,-0.043991,-0.030146,-0.014320,-0.020827,-0.072079,-0.023405,-0.053254
grade:B,-0.281766,1.000000,-0.391304,-0.286559,-0.186528,-0.110100,-0.001790,0.005986,-0.000170,-0.002915,...,-0.036319,0.009747,-0.009788,0.001302,0.001143,-0.009835,-0.004384,-0.015251,-0.003582,0.018410
grade:C,-0.264622,-0.391304,1.000000,-0.269123,-0.175179,-0.103401,0.000212,-0.016001,0.004297,0.001653,...,0.032247,-0.038853,0.029646,0.010905,0.010037,0.009784,0.009233,0.040123,0.009529,0.017134
grade:D,-0.193788,-0.286559,-0.269123,1.000000,-0.128287,-0.075723,0.003670,-0.035240,0.006286,0.001370,...,0.048506,-0.049909,0.032027,0.019243,0.010826,0.009879,0.010159,0.030623,0.011940,0.006545
grade:E,-0.126141,-0.186528,-0.175179,-0.128287,1.000000,-0.049290,0.003561,-0.015168,-0.001328,0.001286,...,0.041550,-0.038756,0.027700,0.010752,0.007767,0.004840,0.007125,0.015033,0.004892,0.002659
grade:F,-0.074456,-0.110100,-0.103401,-0.075723,-0.049290,1.000000,0.000995,-0.014655,0.001718,0.002427,...,0.022871,-0.025030,0.022792,0.003814,0.000479,0.001742,0.000164,-0.000684,0.000945,0.003665
home_ownership:OWN,-0.005355,-0.001790,0.000212,0.003670,0.003561,0.000995,1.000000,-0.316735,0.015010,0.004878,...,0.024337,0.002729,-0.004635,0.001113,0.000265,0.002403,0.001415,0.003611,0.001795,-0.000842
home_ownership:MORTGAGE,0.065822,0.005986,-0.016001,-0.035240,-0.015168,-0.014655,-0.316735,1.000000,-0.000715,0.041385,...,-0.014438,-0.056982,0.052135,0.008760,0.003075,0.006023,-0.000543,-0.014373,-0.004101,0.013401
addr_state:ND_NE_IA_NV_FL_HI_AL,-0.011226,-0.000170,0.004297,0.006286,-0.001328,0.001718,0.015010,-0.000715,1.000000,-0.127503,...,0.016919,-0.002788,0.000973,0.001981,0.001891,-0.008869,-0.009749,0.000375,-0.001867,-0.004992
addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.001681,-0.002915,0.001653,0.001370,0.001286,0.002427,0.004878,0.041385,-0.127503,1.000000,...,0.008275,-0.002663,0.000934,0.000200,0.001951,-0.005617,-0.003450,-0.006139,-0.003866,-0.002917


# PD Model Estimation

## Logistic Regression

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

In [43]:
reg = LogisticRegression()
# We create an instance of an object from the 'LogisticRegression' class.

In [44]:
pd.options.display.max_rows = None
# Sets the pandas dataframe options to display all columns/ rows.

In [45]:
reg.fit(inputs_train, loan_data_targets_train)
# Estimates the coefficients of the object from the 'LogisticRegression' class
# with inputs (independent variables) contained in the first dataframe
# and targets (dependent variables) contained in the second dataframe.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression()

In [46]:
reg.intercept_
# Displays the intercept contain in the estimated ("fitted") object from the 'LogisticRegression' class.

array([-0.23794822])

In [47]:
reg.coef_
# Displays the coefficients contained in the estimated ("fitted") object from the 'LogisticRegression' class.

array([[ 0.91776658,  0.69590012,  0.50025311,  0.33196487,  0.1956864 ,
         0.06780163,  0.0866661 ,  0.12361225, -0.10739435, -0.04164462,
        -0.02416677,  0.00335134,  0.04536689,  0.04584019,  0.108213  ,
         0.18148385,  0.20458932,  0.25389356,  0.59906368,  0.08940471,
        -0.01407579,  0.28370973,  0.17213477,  0.19892856,  0.23614486,
         0.04017614,  0.06217105,  0.12277695,  0.10871407,  0.06777127,
         0.06073242,  0.11799418,  0.9161122 ,  0.79375492,  0.71008392,
         0.53839591,  0.42156338,  0.20658153, -0.02179143,  0.99897184,
         0.6430071 ,  0.39831535,  0.16893927,  0.0857547 ,  0.07099143,
         0.11316473,  0.14987942,  0.1643695 , -0.06922189, -0.11778823,
        -0.02295317,  0.31461707,  0.17886653, -0.00977696,  0.04659772,
        -0.01631672, -0.031007  , -0.05771993, -0.05228246, -0.03670136,
        -0.03249014, -0.00842389,  0.08074442, -0.04991334, -0.09627516,
        -0.00664038,  0.18805794, -0.01213365, -0.0

In [48]:
feature_name = inputs_train.columns.values
# Stores the names of the columns of a dataframe in a variable.

In [49]:
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
# Creates a dataframe with a column titled 'Feature name' and row values contained in the 'feature_name' variable.
summary_table['Coefficients'] = np.transpose(reg.coef_)
# Creates a new column in the dataframe, called 'Coefficients',
# with row values the transposed coefficients from the 'LogisticRegression' object.
summary_table.index = summary_table.index + 1
# Increases the index of every row of the dataframe with 1.
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
# Assigns values of the row with index 0 of the dataframe.
summary_table = summary_table.sort_index()
# Sorts the dataframe by index.
summary_table

,Feature name,Coefficients
0,Intercept,-0.237948
1,grade:A,0.917767
2,grade:B,0.695900
3,grade:C,0.500253
4,grade:D,0.331965
5,grade:E,0.195686
6,grade:F,0.067802
7,home_ownership:OWN,0.086666
8,home_ownership:MORTGAGE,0.123612
9,addr_state:ND_NE_IA_NV_FL_HI_AL,-0.107394


In [50]:
bool_cols = inputs_train.select_dtypes(include=['bool']).columns
inputs_train[bool_cols] = inputs_train[bool_cols].astype(int)

## Build a Logistic Regression Model with P-Values

In [51]:
# P values for sklearn logistic regression.

# Class to display p-values for logistic regression in sklearn.

from sklearn import linear_model
import scipy.stats as stat

class LogisticRegression_with_p_values:

    def __init__(self,*args,**kwargs):#,**kwargs):
        self.model = linear_model.LogisticRegression(*args,**kwargs)#,**args)

    def fit(self,X,y):
        self.model.fit(X,y)

        #### Get p-values for the fitted model ####
        denom = (2.0 * (1.0 + np.cosh(self.model.decision_function(X))))
        denom = np.tile(denom,(X.shape[1],1)).T
        F_ij = np.dot((X / denom).T,X) ## Fisher Information Matrix
        Cramer_Rao = np.linalg.inv(F_ij) ## Inverse Information Matrix
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates # z-score for eaach model coefficient
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores] ### two tailed test for p-values

        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values

In [52]:
reg = LogisticRegression_with_p_values()
# We create an instance of an object from the newly created 'LogisticRegression_with_p_values()' class.

In [53]:
loan_data_targets_train = loan_data_targets_train.values.ravel()

In [54]:
reg.fit(inputs_train, loan_data_targets_train)
# Estimates the coefficients of the object from the 'LogisticRegression' class
# with inputs (independent variables) contained in the first dataframe
# and targets (dependent variables) contained in the second dataframe.

In [55]:
# Same as above.
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table

,Feature name,Coefficients
0,Intercept,-0.237948
1,grade:A,0.917767
2,grade:B,0.695900
3,grade:C,0.500253
4,grade:D,0.331965
5,grade:E,0.195686
6,grade:F,0.067802
7,home_ownership:OWN,0.086666
8,home_ownership:MORTGAGE,0.123612
9,addr_state:ND_NE_IA_NV_FL_HI_AL,-0.107394


In [56]:
# This is a list.
p_values = reg.p_values
# We take the result of the newly added method 'p_values' and store it in a variable 'p_values'.

In [57]:
# Add the intercept for completeness.
p_values = np.append(np.nan, np.array(p_values))
# We add the value 'NaN' in the beginning of the variable with p-values.

In [58]:
summary_table['p_values'] = p_values
# In the 'summary_table' dataframe, we add a new column, called 'p_values', containing the values from the 'p_values' variable.

In [ ]:
inputs_train_with_ref_cat = loan_data_inputs_train.loc[: , ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'home_ownership:OWN',
'home_ownership:MORTGAGE',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'addr_state:NM_VA',
'addr_state:NY',
'addr_state:OK_TN_MO_LA_MD_NC',
'addr_state:CA',
'addr_state:UT_KY_AZ_NJ',
'addr_state:AR_MI_PA_OH_MN',
'addr_state:RI_MA_DE_SD_IN',
'addr_state:GA_WA_OR',
'addr_state:WI_MT',
'addr_state:TX',
'addr_state:IL_CT',
'addr_state:KS_SC_CO_VT_AK_MS',
'addr_state:WV_NH_WY_DC_ME_ID',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'purpose:credit_card',
'purpose:debt_consolidation',
'purpose:oth__med__vacation',
'purpose:major_purch__car__home_impr',
'initial_list_status:f',
'initial_list_status:w',
'term:36',
'term:60',
'emp_length:0',
'emp_length:1',
'emp_length:2-4',
'emp_length:5-6',
'emp_length:7-9',
'emp_length:10',
'mths_since_issue_d:<38',
'mths_since_issue_d:38-39',
'mths_since_issue_d:40-41',
'mths_since_issue_d:42-48',
'mths_since_issue_d:49-52',
'mths_since_issue_d:53-64',
'mths_since_issue_d:65-84',
'mths_since_issue_d:>84',
'int_rate:<9.548',
'int_rate:9.548-12.025',
'int_rate:12.025-15.74',
'int_rate:15.74-20.281',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'mths_since_earliest_cr_line:141-164',
'mths_since_earliest_cr_line:165-247',
'mths_since_earliest_cr_line:248-270',
'mths_since_earliest_cr_line:271-352',
'mths_since_earliest_cr_line:>352',
'inq_last_6mths:0',
'inq_last_6mths:1-2',
'inq_last_6mths:3-6',
'inq_last_6mths:>6',
'acc_now_delinq:0',
'acc_now_delinq:>=1',
'annual_inc:<20K',
'annual_inc:20K-30K',
'annual_inc:30K-40K',
'annual_inc:40K-50K',
'annual_inc:50K-60K',
'annual_inc:60K-70K',
'annual_inc:70K-80K',
'annual_inc:80K-90K',
'annual_inc:90K-100K',
'annual_inc:100K-120K',
'annual_inc:120K-140K',
'annual_inc:>140K',
'dti:<=1.4',
'dti:1.4-3.5',
'dti:3.5-7.7',
'dti:7.7-10.5',
'dti:10.5-16.1',
'dti:16.1-20.3',
'dti:20.3-21.7',
'dti:21.7-22.4',
'dti:22.4-35',
'dti:>35',
'mths_since_last_delinq:Missing',
'mths_since_last_delinq:0-3',
'mths_since_last_delinq:4-30',
'mths_since_last_delinq:31-56',
'mths_since_last_delinq:>=57',
'mths_since_last_record:Missing',
'mths_since_last_record:0-2',
'mths_since_last_record:3-20',
'mths_since_last_record:21-31',
'mths_since_last_record:32-80',
'mths_since_last_record:81-86',
'mths_since_last_record:>=86',
]]

In [ ]:
ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'initial_list_status:f',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'inq_last_6mths:>6',
'acc_now_delinq:0',
'annual_inc:<20K',
'dti:>35',
'mths_since_last_delinq:0-3',
'mths_since_last_record:0-2']

In [ ]:
inputs_train = inputs_train_with_ref_cat.drop(ref_categories, axis = 1)
inputs_train.head()

In [ ]:
# Here we run a new model.
reg2 = LogisticRegression_with_p_values()
reg2.fit(inputs_train, loan_data_targets_train)

In [ ]:
feature_name = inputs_train.columns.values

In [ ]:
# Same as above.
summary_table = pd.DataFrame(columns = ['Feature name'], data = feature_name)
summary_table['Coefficients'] = np.transpose(reg2.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg2.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table

In [ ]:
# We add the 'p_values' here, just as we did before.
p_values = reg2.p_values
p_values = np.append(np.nan,np.array(p_values))
summary_table['p_values'] = p_values
summary_table
# Here we get the results for our final PD model.

In [ ]:
import pickle

In [ ]:
pickle.dump(reg2, open('pd_model.sav', 'wb'))
# Here we export our model to a 'SAV' file with file name 'pd_model.sav'.